# RT-DETRv2 Object Detection on Modal

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/detr-object-detection/tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Fine-tune **RT-DETRv2** (real-time DETR, v2) on a custom COCO-format dataset, evaluate with COCO mAP, and deploy a live detection app — all on [Modal](https://modal.com).

## Setup

See more details at: https://github.com/unionai/workshops/tree/main/tutorials/detr-object-detection

The only local dependency is `modal` — PyTorch, Transformers, and the rest are installed inside the Modal container image, defined in `config.py`.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/detr-object-detection
    !pip install -r requirements.txt

## Connect to Modal

Authenticate once. This opens a browser to link your Modal account (sign up free at [modal.com](https://modal.com)):

In [ ]:
!modal setup

## HuggingFace secret

The pipeline pulls the dataset and base model from HuggingFace. The Modal functions read `HF_TOKEN` from a **Modal secret** named `huggingface-secret`, so gated models and private datasets work wherever the function runs.

Enter your token below (a read-scoped token is fine; the public defaults don't strictly need one, but the secret must exist):

In [ ]:
import os
from getpass import getpass

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')

Store it as a Modal secret so the functions can read it when they run in the cloud (`--force` overwrites an existing secret of the same name):

In [ ]:
!modal secret create huggingface-secret HF_TOKEN=$HF_TOKEN --force

## Run the pipeline

`modal run` builds the image (first run only), then runs data prep on CPU and training/eval on a GPU, streaming logs to your terminal. The `local_entrypoint` writes the fine-tuned model and the styled HTML reports to your working directory when it finishes.

In [ ]:
!modal run workflow.py --epochs 40 --eval-every-n-epochs 10

### Reports

Modal has no built-in live-report UI, so the pipeline builds styled HTML reports and writes them locally. Open them in a browser:

- `training_report.html` — loss chart, LR schedule, periodic mAP
- `evaluation_report.html` — COCO mAP metrics table + bar chart
- `inference_demo.html` — ground truth vs predictions per image
- `pipeline_report.html` — final summary

The fine-tuned model is written to `./finetuned_model/` and saved into the `rtdetr-model` volume for serving.

## Deploy Model & App

Note: wait for the pipeline above to finish before deploying — the server loads the fine-tuned model from the `rtdetr-model` volume.

Deploy the FastAPI model server (persistent URL):

In [ ]:
!modal deploy app_server.py

Deploy the Gradio frontend. It auto-discovers the deployed server and connects to it:

In [ ]:
!modal deploy app_gradio.py

> Tip: while iterating, use `modal serve app_server.py` / `modal serve app_gradio.py` instead — they print an ephemeral URL and hot-reload on file changes.

## Inspect runs in the dashboard

Every `modal run` / `modal deploy` streams logs to your terminal and records the app in the [Modal dashboard](https://modal.com/apps), where you can browse past executions, logs, container metrics, and the deployed app URLs. List your apps with:

```bash
modal app list
```

or visit [modal.com/apps](https://modal.com/apps) directly.